# GeoCLIP baseline on Im2GPS3k (Kaggle)

This notebook attaches `lbgan2000/imgps3k-yfcc4k-cleaned` with `kagglehub`, evaluates the 2,997 Im2GPS3k images that have latitude/longitude labels, and persists both aggregate metrics and per-image predictions. The dataset directory contains 3,000 JPEGs, but its metadata CSV contains only 2,997 labeled rows; unlabeled images are excluded.

In [1]:
import os, kagglehub, json, pathlib, time
handle = "lbgan2000/imgps3k-yfcc4k-cleaned"
t0 = time.time()
dataset_path = kagglehub.dataset_download(handle)
print("dataset_path =", dataset_path)
print("elapsed_sec =", round(time.time() - t0, 2))
print("under_input =", str(dataset_path).startswith("/kaggle/input/"))
root = pathlib.Path(dataset_path)
items = []
for p in root.rglob("*"):
    if p.is_file():
        items.append((str(p.relative_to(root)), p.stat().st_size))
print("file_count =", len(items))
print("total_gib =", round(sum(s for _, s in items) / 2**30, 3))
print("first_files =", items[:20])

Mounting files to /kaggle/input/datasets/lbgan2000/imgps3k-yfcc4k-cleaned...


dataset_path = /kaggle/input/datasets/lbgan2000/imgps3k-yfcc4k-cleaned
elapsed_sec = 45.19
under_input = True


file_count = 12077
total_gib = 1.518
first_files = [('test_set/im2gps3k_places365.csv', 359201), ('test_set/yfcc4k.csv', 2643367), ('test_set/filtered_yfcc4k.csv', 2554686), ('test_set/yfcc4k/yfcc4k.txt', 2613363), ('test_set/yfcc4k_clean/yfcc4k_move.txt', 6620), ('test_set/im2gps3ktest/im2gps3ktest/1259515638_35322982ee_1179_48264126@N00.jpg', 179599), ('test_set/im2gps3ktest/im2gps3ktest/291752231_d0312570a2_113_73091266@N00.jpg', 101804), ('test_set/im2gps3ktest/im2gps3ktest/314039220_7045707399_115_26519935@N00.jpg', 110488), ('test_set/im2gps3ktest/im2gps3ktest/1086319092_34f919b886_1232_33463080@N00.jpg', 309073), ('test_set/im2gps3ktest/im2gps3ktest/349388449_84a9d2f8b5_128_51162504@N00.jpg', 126840), ('test_set/im2gps3ktest/im2gps3ktest/396111553_d5bff35e7d_150_14542551@N00.jpg', 128107), ('test_set/im2gps3ktest/im2gps3ktest/348424445_99ce2d52aa_146_85971448@N00.jpg', 99761), ('test_set/im2gps3ktest/im2gps3ktest/158955434_7da6c487d6_51_18684820@N00.jpg', 109703), ('test_set/im2

In [2]:
import pandas as pd, pathlib, os
root = pathlib.Path(dataset_path)
csv_path = root / "test_set" / "im2gps3k_places365.csv"
df = pd.read_csv(csv_path)
print("shape =", df.shape)
print("columns =", df.columns.tolist())
print(df.head(3).to_string())
image_dir = root / "test_set" / "im2gps3ktest" / "im2gps3ktest"
images = list(image_dir.glob("*"))
print("image_count =", len(images))
print("sample_image_names =", [p.name for p in images[:5]])
print("dtypes =", df.dtypes.astype(str).to_dict())
print("null_counts =", df.isna().sum().to_dict())

shape = (2997, 10)
columns = ['name', 'AUTHOR', 'LAT', 'LON', 'S3_Label', 'S16_Label', 'S365_Label', 'Prob_indoor', 'Prob_natural', 'Prob_urban']
                                          name        AUTHOR        LAT        LON  S3_Label  S16_Label  S365_Label  Prob_indoor  Prob_natural  Prob_urban
0  1000269685_e60e9cdfb4_1125_78841376@N00.jpg  78841376@N00  32.325436 -64.764404         2         12         353     0.274242      0.045113    0.680645
1  1000304467_1a75a200b1_1296_78841376@N00.jpg  78841376@N00  32.325436 -64.764404         0          4         325     0.414407      0.220912    0.364681
2  1001048550_8e4b47d165_1051_78841376@N00.jpg  78841376@N00  32.325436 -64.764404         1          8          36     0.007326      0.969903    0.022771
image_count = 3000
sample_image_names = ['1259515638_35322982ee_1179_48264126@N00.jpg', '291752231_d0312570a2_113_73091266@N00.jpg', '314039220_7045707399_115_26519935@N00.jpg', '1086319092_34f919b886_1232_33463080@N00.jpg', '34938844

In [4]:
import os, sys, time, json, math
from pathlib import Path
import numpy as np
import torch

base = Path("/kaggle/working/geoclip_baseline")
source_dir = base / "source"
hf_root = base / "hf"
model_file = hf_root / "hub/models--openai--clip-vit-large-patch14/snapshots/local/model.safetensors"
assert source_dir.is_dir(), f"Missing GeoCLIP source: {source_dir}"
assert model_file.is_file(), f"Missing offline CLIP backbone: {model_file}"

os.environ.update({
    "HF_HOME": str(hf_root),
    "HF_HUB_CACHE": str(hf_root / "hub"),
    "TRANSFORMERS_OFFLINE": "1",
    "HF_HUB_OFFLINE": "1",
})
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from geoclip import GeoCLIP
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
model = GeoCLIP().to(device).eval()
with torch.inference_mode():
    gallery_gpu = model.gps_gallery.to(device)
    gallery_features = F.normalize(model.location_encoder(gallery_gpu), dim=1)
    logit_scale = model.logit_scale.exp()
if device == "cuda":
    torch.cuda.synchronize()
setup_seconds = time.perf_counter() - t0
print("GEOCLIP_SETUP_OK")
print("device =", device)
print("gpu =", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
print("gallery_shape =", tuple(model.gps_gallery.shape))
print("gallery_features_shape =", tuple(gallery_features.shape))
print("setup_seconds =", round(setup_seconds, 3))
print("peak_gpu_gib =", round(torch.cuda.max_memory_allocated()/2**30, 3) if device == "cuda" else 0)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


GEOCLIP_SETUP_OK
device = cuda
gpu = NVIDIA RTX PRO 6000 Blackwell Server Edition
gallery_shape = (100000, 2)
gallery_features_shape = (100000, 512)
setup_seconds = 1.576
peak_gpu_gib = 2.598


In [5]:
from PIL import Image, ImageFile
from transformers import AutoProcessor
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Reproduce the preprocessing used by the original GeoCLIP release.
model.image_encoder.image_processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-large-patch14",
    use_fast=False,
    local_files_only=True,
)

eval_df = df.copy()
eval_df["image_path"] = eval_df["name"].map(lambda x: str(image_dir / x))
missing = eval_df.loc[~eval_df["image_path"].map(os.path.isfile), "name"].tolist()
assert not missing, f"Missing labeled images: {missing[:10]}"

batch_size = 128
pred_indices = []
failed = []
t0 = time.perf_counter()

with torch.inference_mode():
    for start in range(0, len(eval_df), batch_size):
        batch = eval_df.iloc[start:start + batch_size]
        pil_images = []
        valid_positions = []
        for pos, image_path in enumerate(batch["image_path"]):
            try:
                with Image.open(image_path) as im:
                    pil_images.append(im.convert("RGB"))
                valid_positions.append(pos)
            except Exception as exc:
                failed.append({"name": Path(image_path).name, "error": repr(exc)})
        if len(valid_positions) != len(batch):
            raise RuntimeError(f"Unreadable benchmark images: {failed[-5:]}")
        pixel_values = model.image_encoder.image_processor(
            images=pil_images, return_tensors="pt"
        )["pixel_values"].to(device)
        image_features = F.normalize(model.image_encoder(pixel_values), dim=1)
        logits = logit_scale * (image_features @ gallery_features.T)
        pred_indices.extend(logits.argmax(dim=1).cpu().tolist())
        if device == "cuda":
            torch.cuda.synchronize()
        done = min(start + batch_size, len(eval_df))
        if done == len(eval_df) or done % (batch_size * 5) == 0:
            print(f"processed {done}/{len(eval_df)}")

inference_seconds = time.perf_counter() - t0
pred_indices = np.asarray(pred_indices)
pred_gps = model.gps_gallery[pred_indices].cpu().numpy()
target_gps = eval_df[["LAT", "LON"]].to_numpy(dtype=np.float64)

lat1, lon1 = np.radians(target_gps[:, 0]), np.radians(target_gps[:, 1])
lat2, lon2 = np.radians(pred_gps[:, 0]), np.radians(pred_gps[:, 1])
dlat, dlon = lat2 - lat1, lon2 - lon1
a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
distances_km = 6371.0088 * 2 * np.arctan2(np.sqrt(a), np.sqrt(np.maximum(0, 1-a)))

thresholds = [1, 25, 200, 750, 2500]
accuracies = {f"acc_{d}_km": float(np.mean(distances_km <= d)) for d in thresholds}
result = {
    "dataset": handle,
    "dataset_path": str(dataset_path),
    "split": "im2gps3k",
    "labeled_images": int(len(eval_df)),
    "evaluated_images": int(len(pred_indices)),
    "failed_images": failed,
    "gallery_size": int(len(model.gps_gallery)),
    "device": device,
    "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
    "batch_size": batch_size,
    "setup_seconds": setup_seconds,
    "inference_seconds": inference_seconds,
    "images_per_second": len(eval_df) / inference_seconds,
    "mean_distance_km": float(np.mean(distances_km)),
    "median_distance_km": float(np.median(distances_km)),
    "accuracy": accuracies,
    "peak_gpu_gib": torch.cuda.max_memory_allocated()/2**30 if device == "cuda" else 0.0,
}
pred_table = eval_df[["name", "LAT", "LON"]].copy()
pred_table["pred_lat"] = pred_gps[:, 0]
pred_table["pred_lon"] = pred_gps[:, 1]
pred_table["distance_km"] = distances_km

out_dir = base / "im2gps3k_eval"
out_dir.mkdir(parents=True, exist_ok=True)
summary_path = out_dir / "summary.json"
predictions_path = out_dir / "predictions.csv"
summary_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
pred_table.to_csv(predictions_path, index=False)

print(json.dumps(result, indent=2))
print("summary_saved =", summary_path)
print("predictions_saved =", predictions_path)

processed 640/2997


processed 1280/2997


processed 1920/2997


processed 2560/2997


processed 2997/2997
{
  "dataset": "lbgan2000/imgps3k-yfcc4k-cleaned",
  "dataset_path": "/kaggle/input/datasets/lbgan2000/imgps3k-yfcc4k-cleaned",
  "split": "im2gps3k",
  "labeled_images": 2997,
  "evaluated_images": 2997,
  "failed_images": [],
  "gallery_size": 100000,
  "device": "cuda",
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "batch_size": 128,
  "setup_seconds": 1.5762174259998574,
  "inference_seconds": 47.09041775000014,
  "images_per_second": 63.6435211917395,
  "mean_distance_km": 1763.001525761393,
  "median_distance_km": 241.78200745314317,
  "accuracy": {
    "acc_1_km": 0.1304637971304638,
    "acc_25_km": 0.321654988321655,
    "acc_200_km": 0.47981314647981316,
    "acc_750_km": 0.665331998665332,
    "acc_2500_km": 0.8228228228228228
  },
  "peak_gpu_gib": 3.9617433547973633
}
summary_saved = /kaggle/working/geoclip_baseline/im2gps3k_eval/summary.json
predictions_saved = /kaggle/working/geoclip_baseline/im2gps3k_eval/predictions.csv
